# Projection Test Notebook

This notebook tests projection functions by:
1. Loading two consecutive frames
2. Extracting 3D points from frame 2
3. Projecting them to frame 1 using relative pose
4. Checking if projected points are inside the segmentation mask
5. Checking if depth values are consistent

In [ ]:
# Force refresh
import os
import sys

sys.path.append("../")
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from omegaconf import OmegaConf
import glob

from point2pose.data_types.frame import Frame
from point2pose.pipeline.modular_pipeline import ModularPipeline
from point2pose.utils.camera import (
    extract_cropped_point_cloud, 
    project_points_to_image, 
    convert_pixel_to_world,
    compute_projection_consistency
)
from point2pose.utils.transform import transform_pts, inverse_SE3
from point2pose.io.sources.dataset.datareader import Ho3dReader

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Option 1: Load from HO3D Dataset

In [ ]:
# Load frames from HO3D dataset
# Adjust these paths to your data location
ho3d_root = "/home/justin/data/HO3D_V3/"  # Root directory of HO3D dataset
video_dir = "/home/justin/data/HO3D_V3/evaluation/MPM10"  # Path to specific video sequence

# Initialize reader
reader = Ho3dReader(video_dir, ho3d_root)
print(f"Loaded {len(reader)} frames from {reader.get_video_name()}")

# Load two consecutive frames
frame_idx_0 = 260
frame_idx_1 = 261

# Load frame 0
rgb_0 = cv2.cvtColor(cv2.imread(reader.color_files[frame_idx_0]), cv2.COLOR_BGR2RGB)
depth_0 = reader.get_depth(frame_idx_0)
mask_0 = reader.get_mask(frame_idx_0)
H, W = rgb_0.shape[:2]
mask_0 = cv2.resize(mask_0, (W, H), interpolation=cv2.INTER_NEAREST)
mask_0_tensor = torch.from_numpy(mask_0).float().unsqueeze(0).unsqueeze(0).to(device)

frame_0 = Frame(
    id=frame_idx_0,
    rgb=rgb_0,
    depth=depth_0,
    mask=mask_0_tensor,
    intrinsics=reader.K,
    depth_factor=1.0,  # HO3D depth is already in meters
)

# Load frame 1
rgb_1 = cv2.cvtColor(cv2.imread(reader.color_files[frame_idx_1]), cv2.COLOR_BGR2RGB)
depth_1 = reader.get_depth(frame_idx_1)
mask_1 = reader.get_mask(frame_idx_1)
mask_1 = cv2.resize(mask_1, (W, H), interpolation=cv2.INTER_NEAREST)
mask_1_tensor = torch.from_numpy(mask_1).float().unsqueeze(0).unsqueeze(0).to(device)

frame_1 = Frame(
    id=frame_idx_1,
    rgb=rgb_1,
    depth=depth_1,
    mask=mask_1_tensor,
    intrinsics=reader.K,
    depth_factor=1.0,
)

print(f"Frame 0: RGB {frame_0.rgb.shape}, Depth {frame_0.depth.shape}, Mask {frame_0.mask.shape}")
print(f"Frame 1: RGB {frame_1.rgb.shape}, Depth {frame_1.depth.shape}, Mask {frame_1.mask.shape}")

## Get Poses (Using Pipeline or GT)

In [ ]:
# Option A: Use pipeline to estimate poses
config_path = "../configs/ho3d/ho3d_single.yaml"
cfg = OmegaConf.load(config_path)
pipeline = ModularPipeline(cfg)

# Process frames through pipeline
pipeline.step(frame_0)
pose_0 = pipeline.objects[0].pose.copy() if pipeline.objects[0].pose is not None else np.eye(4)

pipeline.step(frame_1)
pose_1 = pipeline.objects[0].pose.copy() if pipeline.objects[0].pose is not None else np.eye(4)

print("Poses from pipeline:")
print(f"Frame 0 pose:\n{pose_0}")
print(f"\nFrame 1 pose:\n{pose_1}")

In [ ]:
# Option B: Use GT poses from reader (if available)
gt_pose_0 = reader.get_gt_pose(frame_idx_0)
gt_pose_1 = reader.get_gt_pose(frame_idx_1)

if gt_pose_0 is not None and gt_pose_1 is not None:
    print("Using GT poses:")
    print(f"Frame 0 GT pose:\n{gt_pose_0}")
    print(f"\nFrame 1 GT pose:\n{gt_pose_1}")
    
    # Use GT poses instead
    pose_0 = gt_pose_0
    pose_1 = gt_pose_1
else:
    print("GT poses not available, using pipeline poses")

## Projection Consistency Test Functions

The projection consistency checking is split into several modular functions:
- `extract_and_project_points`: Extracts 3D points and projects them
- `check_mask_consistency`: Checks if projected points are inside the mask
- `compute_depth_consistency_score`: Computes a depth consistency score for each point
- `evaluate_projection_consistency`: Evaluates all points and returns per-point results
- `compute_statistics`: Computes aggregate statistics
- `visualize_projection_results`: Creates visualizations
- `check_projection_consistency`: Main orchestrator function

In [ ]:
def extract_and_project_points(frame_src, frame_dst, T_dst_src, obj_id=0, 
                                min_depth_extract=0.05, max_depth_extract=0.8):
    """
    Extract 3D points from source frame and project them to destination frame.
    
    Args:
        frame_src: Source Frame object
        frame_dst: Destination Frame object
        T_dst_src: Transform from Source camera to Destination camera
        obj_id: Object ID to extract
        min_depth_extract: Minimum depth for point extraction (meters)
        max_depth_extract: Maximum depth for point extraction (meters)
        
    Returns:
        tuple: (pts_src, pts_dst_2d, pts_dst_3d) or None if extraction fails
    """
    # Extract 3D points from source frame
    pts_src = extract_cropped_point_cloud(
        frame_src, 
        obj_id=obj_id, 
        min_depth=min_depth_extract,
        max_depth=max_depth_extract
    )
    
    if len(pts_src) == 0:
        print("No points extracted from source frame.")
        return None
    
    print(f"Extracted {len(pts_src)} points from source frame.")

    # Project points to destination frame
    pts_dst_2d, pts_dst_3d = project_points_to_image(
        pts_src, 
        frame_dst.intrinsics, 
        T_dst_src
    )
    
    return pts_src, pts_dst_2d, pts_dst_3d


def check_mask_consistency(pts_2d, mask, H, W):
    """
    Check if projected points are inside the segmentation mask.
    
    Args:
        pts_2d: (N, 2) array of 2D pixel coordinates
        mask: (H, W) binary mask
        H: Image height
        W: Image width
        
    Returns:
        dict with keys: 'in_bounds', 'inside_mask', 'outside_mask', 'mask_status'
        where mask_status is a boolean array indicating if each point is inside mask
    """
    mask_status = np.zeros(len(pts_2d), dtype=bool)
    in_bounds = np.zeros(len(pts_2d), dtype=bool)
    
    for i, (u, v) in enumerate(pts_2d):
        u_int, v_int = int(round(u)), int(round(v))
        
        # Check bounds
        if 0 <= u_int < W and 0 <= v_int < H:
            in_bounds[i] = True
            # Check inside mask
            if mask[v_int, u_int] > 0:
                mask_status[i] = True
    
    return {
        'in_bounds': in_bounds,
        'inside_mask': mask_status,
        'outside_mask': in_bounds & ~mask_status,
        'mask_status': mask_status
    }


def compute_depth_consistency_score(z_projected, z_measured, depth_tolerance=0.02, 
                                     score_type='linear'):
    """
    Compute a depth consistency score for a single point.
    
    Args:
        z_projected: Projected depth value (meters)
        z_measured: Measured depth value from depth image (meters)
        depth_tolerance: Maximum allowed depth difference for perfect score (meters)
        score_type: Type of scoring function ('linear', 'exponential', 'gaussian')
        
    Returns:
        tuple: (score, depth_error) where:
            - score: float in [0, 1], higher is better (1.0 = perfect match)
            - depth_error: absolute depth difference in meters
    """
    depth_error = abs(z_projected - z_measured)
    
    if score_type == 'linear':
        # Linear decay from 1.0 at error=0 to 0.0 at error=tolerance
        score = max(0.0, 1.0 - (depth_error / depth_tolerance))
    elif score_type == 'exponential':
        # Exponential decay: score = exp(-error / (tolerance / 3))
        score = np.exp(-depth_error / (depth_tolerance / 3.0))
    elif score_type == 'gaussian':
        # Gaussian: score = exp(-0.5 * (error / (tolerance / 2))^2)
        sigma = depth_tolerance / 2.0
        score = np.exp(-0.5 * (depth_error / sigma) ** 2)
    else:
        raise ValueError(f"Unknown score_type: {score_type}")
    
    return score, depth_error


def evaluate_projection_consistency(pts_dst_2d, pts_dst_3d, frame_dst, mask_status, 
                                     obj_id=0, depth_tolerance=0.02, 
                                     min_depth=0.01, max_depth=2.0,
                                     score_type='linear'):
    """
    Evaluate projection consistency for all points.
    
    Args:
        pts_dst_2d: (N, 2) array of projected 2D coordinates
        pts_dst_3d: (N, 3) array of 3D points in destination frame
        frame_dst: Destination Frame object
        mask_status: (N,) boolean array indicating mask consistency
        obj_id: Object ID
        depth_tolerance: Maximum allowed depth difference (meters)
        min_depth: Minimum valid depth (meters) - points outside this range are filtered
        max_depth: Maximum valid depth (meters) - points outside this range are filtered
        score_type: Type of scoring function for depth consistency
        
    Returns:
        dict with per-point evaluation results
    """
    H, W = frame_dst.depth.shape
    N = len(pts_dst_2d)
    
    results = {
        'valid': np.zeros(N, dtype=bool),
        'in_bounds': np.zeros(N, dtype=bool),
        'inside_mask': mask_status.copy(),
        'depth_valid': np.zeros(N, dtype=bool),
        'depth_in_range': np.zeros(N, dtype=bool),
        'depth_error': np.full(N, np.nan, dtype=float),
        'depth_score': np.full(N, np.nan, dtype=float),
        'depth_consistent': np.zeros(N, dtype=bool),
    }
    
    for i, (u, v) in enumerate(pts_dst_2d):
        u_int, v_int = int(round(u)), int(round(v))
        
        # Check bounds
        if 0 <= u_int < W and 0 <= v_int < H:
            results['in_bounds'][i] = True
            
            # Get measured depth
            z_measured = frame_dst.depth[v_int, u_int] / frame_dst.depth_factor
            z_projected = pts_dst_3d[i, 2]
            
            # Check if depth is valid and in range
            if z_measured > 0 and np.isfinite(z_measured):
                results['depth_valid'][i] = True
                
                # Check if depth is within valid range
                if min_depth <= z_measured <= max_depth and min_depth <= z_projected <= max_depth:
                    results['depth_in_range'][i] = True
                    
                    # Only evaluate depth consistency if point is inside mask and depth is in range
                    if mask_status[i] and results['depth_in_range'][i]:
                        score, depth_error = compute_depth_consistency_score(
                            z_projected, z_measured, depth_tolerance, score_type
                        )
                        results['depth_error'][i] = depth_error
                        results['depth_score'][i] = score
                        results['depth_consistent'][i] = (depth_error < depth_tolerance)
                        results['valid'][i] = True
    
    return results


def compute_statistics(results, depth_tolerance=0.02):
    """
    Compute aggregate statistics from per-point evaluation results.
    
    Args:
        results: dict from evaluate_projection_consistency
        depth_tolerance: Depth tolerance for consistency check (meters)
        
    Returns:
        dict with aggregate statistics
    """
    N = len(results['valid'])
    
    stats = {
        'total_points': N,
        'in_bounds': np.sum(results['in_bounds']),
        'inside_mask': np.sum(results['inside_mask']),
        'outside_mask': np.sum(results['in_bounds'] & ~results['inside_mask']),
        'depth_valid': np.sum(results['depth_valid']),
        'depth_in_range': np.sum(results['depth_in_range']),
        'valid_evaluations': np.sum(results['valid']),
        'depth_consistent': np.sum(results['depth_consistent']),
        'depth_inconsistent': np.sum(results['valid'] & ~results['depth_consistent']),
    }
    
    # Compute depth error statistics
    valid_depth_errors = results['depth_error'][results['valid']]
    valid_depth_scores = results['depth_score'][results['valid']]
    
    if len(valid_depth_errors) > 0:
        stats['depth_errors'] = valid_depth_errors.tolist()
        stats['depth_scores'] = valid_depth_scores.tolist()
        stats['mean_depth_error'] = np.mean(valid_depth_errors)
        stats['median_depth_error'] = np.median(valid_depth_errors)
        stats['max_depth_error'] = np.max(valid_depth_errors)
        stats['std_depth_error'] = np.std(valid_depth_errors)
        stats['mean_depth_score'] = np.mean(valid_depth_scores)
        stats['min_depth_score'] = np.min(valid_depth_scores)
    else:
        stats['depth_errors'] = []
        stats['depth_scores'] = []
        stats['mean_depth_error'] = np.nan
        stats['median_depth_error'] = np.nan
        stats['max_depth_error'] = np.nan
        stats['std_depth_error'] = np.nan
        stats['mean_depth_score'] = np.nan
        stats['min_depth_score'] = np.nan
    
    return stats


def visualize_projection_results(frame_dst, pts_dst_2d, results, stats, depth_tolerance=0.02):
    """
    Create visualizations of projection consistency results.
    
    Args:
        frame_dst: Destination Frame object
        pts_dst_2d: (N, 2) array of projected 2D coordinates
        results: dict from evaluate_projection_consistency
        stats: dict from compute_statistics
        depth_tolerance: Depth tolerance for visualization (meters)
    """
    # Prepare visualization data
    inside_mask_points = pts_dst_2d[results['in_bounds'] & results['inside_mask']]
    outside_mask_points = pts_dst_2d[results['in_bounds'] & ~results['inside_mask']]
    
    valid_indices = results['valid']
    depth_consistent_points = pts_dst_2d[valid_indices & results['depth_consistent']]
    depth_inconsistent_points = pts_dst_2d[valid_indices & ~results['depth_consistent']]
    depth_inconsistent_errors = results['depth_error'][valid_indices & ~results['depth_consistent']]
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Plot 1: All projected points colored by mask status
    axes[0].imshow(frame_dst.rgb)
    if len(inside_mask_points) > 0:
        axes[0].scatter(inside_mask_points[:, 0], inside_mask_points[:, 1], 
                      c='green', s=2, alpha=0.6, 
                      label=f'Inside mask ({len(inside_mask_points)})')
    if len(outside_mask_points) > 0:
        axes[0].scatter(outside_mask_points[:, 0], outside_mask_points[:, 1], 
                      c='red', s=2, alpha=0.6, 
                      label=f'Outside mask ({len(outside_mask_points)})')
    axes[0].set_title("Projected Points: Mask Consistency")
    axes[0].legend()
    axes[0].axis('off')
    
    # Plot 2: Points inside mask colored by depth consistency
    axes[1].imshow(frame_dst.rgb)
    if len(depth_consistent_points) > 0:
        axes[1].scatter(depth_consistent_points[:, 0], depth_consistent_points[:, 1], 
                       c='green', s=2, alpha=0.6, 
                       label=f'Depth consistent ({len(depth_consistent_points)})')
    if len(depth_inconsistent_points) > 0:
        scatter = axes[1].scatter(depth_inconsistent_points[:, 0], 
                                 depth_inconsistent_points[:, 1], 
                                 c=depth_inconsistent_errors*1000, s=2, alpha=0.6, 
                                 cmap='Reds', vmin=0, vmax=depth_tolerance*2000,
                                 label=f'Depth inconsistent ({len(depth_inconsistent_points)})')
        plt.colorbar(scatter, ax=axes[1], label='Depth error (mm)')
    axes[1].set_title("Projected Points: Depth Consistency")
    axes[1].legend()
    axes[1].axis('off')
    
    # Plot 3: Depth error histogram
    if len(stats['depth_errors']) > 0:
        axes[2].hist(np.array(stats['depth_errors'])*1000, bins=50, edgecolor='black', alpha=0.7)
        axes[2].axvline(depth_tolerance*1000, color='r', linestyle='--', 
                       label=f'Tolerance ({depth_tolerance*1000:.1f}mm)')
        axes[2].set_xlabel('Depth Error (mm)')
        axes[2].set_ylabel('Frequency')
        axes[2].set_title('Depth Error Distribution')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, 'No depth errors\nto display', 
                     ha='center', va='center', transform=axes[2].transAxes)
        axes[2].set_title('Depth Error Distribution')
    
    plt.tight_layout()
    plt.show()


def print_statistics(stats, depth_tolerance=0.02):
    """
    Print projection consistency statistics.
    
    Args:
        stats: dict from compute_statistics
        depth_tolerance: Depth tolerance (meters)
    """
    print("\n" + "="*60)
    print("PROJECTION CONSISTENCY RESULTS")
    print("="*60)
    print(f"Total points extracted: {stats['total_points']}")
    print(f"Points in bounds: {stats['in_bounds']} ({stats['in_bounds']/stats['total_points']*100:.1f}%)")
    print(f"Points inside mask: {stats['inside_mask']} ({stats['inside_mask']/stats['total_points']*100:.1f}%)")
    print(f"Points outside mask: {stats['outside_mask']} ({stats['outside_mask']/stats['total_points']*100:.1f}%)")
    print(f"Depth valid: {stats['depth_valid']} ({stats['depth_valid']/stats['total_points']*100:.1f}%)")
    print(f"Depth in range: {stats['depth_in_range']} ({stats['depth_in_range']/stats['total_points']*100:.1f}%)")
    print(f"Valid evaluations: {stats['valid_evaluations']} ({stats['valid_evaluations']/stats['total_points']*100:.1f}%)")
    
    if stats['valid_evaluations'] > 0:
        print(f"\nDepth Consistency (tolerance: {depth_tolerance*1000:.1f}mm):")
        print(f"  Consistent: {stats['depth_consistent']} ({stats['depth_consistent']/stats['valid_evaluations']*100:.1f}%)")
        print(f"  Inconsistent: {stats['depth_inconsistent']} ({stats['depth_inconsistent']/stats['valid_evaluations']*100:.1f}%)")
        print(f"  Mean depth error: {stats['mean_depth_error']*1000:.2f} mm")
        print(f"  Median depth error: {stats['median_depth_error']*1000:.2f} mm")
        print(f"  Max depth error: {stats['max_depth_error']*1000:.2f} mm")
        print(f"  Std depth error: {stats['std_depth_error']*1000:.2f} mm")
        print(f"  Mean depth score: {stats['mean_depth_score']:.3f}")
        print(f"  Min depth score: {stats['min_depth_score']:.3f}")
    else:
        print("\nNo valid depth comparisons possible.")
    
    print("="*60 + "\n")


def check_projection_consistency(frame_src, frame_dst, T_dst_src, obj_id=0, 
                                  depth_tolerance=0.02, min_depth=0.01, max_depth=2.0,
                                  min_depth_extract=0.01, max_depth_extract=2.0,
                                  score_type='linear', visualize=True, verbose=True):
    """
    Main function to check projection consistency between two frames.
    
    Args:
        frame_src: Source Frame object (e.g., Frame 2, index 1)
        frame_dst: Destination Frame object (e.g., Frame 1, index 0)
        T_dst_src: Transform from Source camera to Destination camera (T_cam_dst_cam_src)
        obj_id: Object ID to extract
        depth_tolerance: Maximum allowed depth difference for consistency (meters)
        min_depth: Minimum valid depth for evaluation (meters) - filters out points outside range
        max_depth: Maximum valid depth for evaluation (meters) - filters out points outside range
        min_depth_extract: Minimum depth for point extraction (meters)
        max_depth_extract: Maximum depth for point extraction (meters)
        score_type: Type of scoring function ('linear', 'exponential', 'gaussian')
        visualize: Whether to create visualizations
        verbose: Whether to print statistics
        
    Returns:
        dict with keys:
            - 'results': per-point evaluation results
            - 'stats': aggregate statistics
            - 'points': (pts_src, pts_dst_2d, pts_dst_3d)
    """
    # 1. Extract and project points
    projection_data = extract_and_project_points(
        frame_src, frame_dst, T_dst_src, obj_id, 
        min_depth_extract, max_depth_extract
    )
    
    if projection_data is None:
        return None
    
    pts_src, pts_dst_2d, pts_dst_3d = projection_data
    
    # 2. Check mask consistency
    H, W = frame_dst.depth.shape
    dst_mask = frame_dst.mask[obj_id, 0].cpu().numpy()
    mask_results = check_mask_consistency(pts_dst_2d, dst_mask, H, W)
    
    # 3. Evaluate projection consistency
    results = evaluate_projection_consistency(
        pts_dst_2d, pts_dst_3d, frame_dst, mask_results['mask_status'],
        obj_id, depth_tolerance, min_depth, max_depth, score_type
    )
    
    # 4. Compute statistics
    stats = compute_statistics(results, depth_tolerance)
    
    # 5. Print statistics
    if verbose:
        print_statistics(stats, depth_tolerance)
    
    # 6. Visualize
    if visualize:
        visualize_projection_results(frame_dst, pts_dst_2d, results, stats, depth_tolerance)
    
    return {
        'results': results,
        'stats': stats,
        'points': (pts_src, pts_dst_2d, pts_dst_3d)
    }

## Run Projection Test

In [ ]:
# Compute Relative Pose T_c0_c1 (from Frame 1 camera to Frame 0 camera)
# We have object poses in each camera frame:
#   P_0 = T_c0_obj @ P_obj  (points in camera 0 frame)
#   P_1 = T_c1_obj @ P_obj  (points in camera 1 frame)
# 
# To transform from camera 1 to camera 0:
#   P_obj = inv(T_c1_obj) @ P_1
#   P_0 = T_c0_obj @ inv(T_c1_obj) @ P_1
#   Therefore: T_c0_c1 = T_c0_obj @ inv(T_c1_obj)

T_c0_obj = pose_0  # Object pose in camera 0 frame
T_c1_obj = pose_1  # Object pose in camera 1 frame

T_c0_c1 = T_c0_obj @ np.linalg.inv(T_c1_obj)

print("Testing projection from Frame 2 (index 1) to Frame 1 (index 0)...")
print(f"Frame 0 pose:\n{T_c0_obj}")
print(f"\nFrame 1 pose:\n{T_c1_obj}")
print(f"\nRelative transform T_c0_c1:\n{T_c0_c1}")

# Run projection consistency check
# You can adjust these parameters:
# - min_depth, max_depth: Filter points outside this depth range for evaluation
# - min_depth_extract, max_depth_extract: Depth range for initial point extraction
# - depth_tolerance: Maximum allowed depth difference for consistency (meters)
# - score_type: 'linear', 'exponential', or 'gaussian' for depth scoring
result = check_projection_consistency(
    frame_1, frame_0, T_c0_c1, 
    obj_id=0, 
    depth_tolerance=0.02,
    min_depth=0.01,      # Minimum depth for evaluation (filters out points)
    max_depth=2.0,       # Maximum depth for evaluation (filters out points)
    min_depth_extract=0.01,  # Minimum depth for extraction
    max_depth_extract=2.0,   # Maximum depth for extraction
    score_type='linear',     # 'linear', 'exponential', or 'gaussian'
    visualize=True,
    verbose=True
)

# Access results
if result is not None:
    results = result['results']  # Per-point evaluation results
    stats = result['stats']      # Aggregate statistics
    pts_src, pts_dst_2d, pts_dst_3d = result['points']  # Point arrays
    
    # Example: Access per-point depth scores
    print(f"\nExample: First 10 depth scores:")
    valid_scores = results['depth_score'][results['valid']]
    if len(valid_scores) > 0:
        print(valid_scores[:10])

## Test compute_projection_consistency Function

Test the new vectorized `compute_projection_consistency` function from `camera.py` that returns mean depth error.

In [ ]:
# Test the new compute_projection_consistency function
# This function is vectorized and returns mean depth error directly

# First, extract points from source frame (frame_1)
pts_src = extract_cropped_point_cloud(
    frame_1, 
    obj_id=0, 
    min_depth=0.01,
    max_depth=2.0
)

print(f"Extracted {len(pts_src)} points from source frame (frame_1)")

if len(pts_src) > 0:
    # Test compute_projection_consistency
    # This function projects points and computes mean depth error
    mean_depth_error = compute_projection_consistency(
        src_pcd=pts_src,
        T_src2dst=T_c0_c1,  # Transform from frame_1 to frame_0
        frame_dst=frame_0,   # Destination frame
        obj_id=0,
        min_depth=0.01,
        max_depth=2.0
    )
    
    print(f"\n{'='*60}")
    print("COMPUTE_PROJECTION_CONSISTENCY RESULTS")
    print(f"{'='*60}")
    print(f"Mean depth error: {mean_depth_error*1000:.2f} mm")
    
    if np.isfinite(mean_depth_error):
        print(f"✓ Projection consistency check passed")
        print(f"  The lower the error, the better the projection consistency")
    else:
        print(f"✗ No valid points found for projection consistency check")
        print(f"  This could mean:")
        print(f"    - Points are outside image bounds")
        print(f"    - Points are outside the segmentation mask")
        print(f"    - Depth values are invalid or outside range")
    
    print(f"{'='*60}\n")
    
    # Compare with the detailed check_projection_consistency if available
    if 'result' in locals() and result is not None:
        detailed_stats = result['stats']
        if 'mean_depth_error' in detailed_stats and np.isfinite(detailed_stats['mean_depth_error']):
            detailed_mean = detailed_stats['mean_depth_error']
            print(f"Comparison with detailed function:")
            print(f"  compute_projection_consistency: {mean_depth_error*1000:.2f} mm")
            print(f"  check_projection_consistency:    {detailed_mean*1000:.2f} mm")
            diff = abs(mean_depth_error - detailed_mean)
            print(f"  Difference: {diff*1000:.2f} mm")
            if diff < 1e-6:
                print(f"  ✓ Results match!")
            else:
                print(f"  ⚠ Small difference (may be due to different filtering)")
else:
    print("No points extracted from source frame. Cannot test compute_projection_consistency.")

## Additional Analysis: Compare Source and Destination Frames

In [ ]:
# Visualize source and destination frames side by side
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# Frame 0 (destination)
axes[0, 0].imshow(frame_0.rgb)
axes[0, 0].set_title(f"Frame 0 (Destination) - RGB")
axes[0, 0].axis('off')

axes[0, 1].imshow(frame_0.mask[0, 0].cpu().numpy(), cmap='gray')
axes[0, 1].set_title(f"Frame 0 (Destination) - Mask")
axes[0, 1].axis('off')

# Frame 1 (source)
axes[1, 0].imshow(frame_1.rgb)
axes[1, 0].set_title(f"Frame 1 (Source) - RGB")
axes[1, 0].axis('off')

axes[1, 1].imshow(frame_1.mask[0, 0].cpu().numpy(), cmap='gray')
axes[1, 1].set_title(f"Frame 1 (Source) - Mask")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()